# PGA Tournament Model: Shot Distance and Locations

Multi-Task learning model. Model needs to output shot distance (float) and shot location (class) for every players shot in the test dataset.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## Get Data

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "tournament_shots_data.csv"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "samanthastaheli/tournamentshots/versions/9",
  file_path,
  pandas_kwargs={"encoding": "latin1"} # European players have different spelling
)

df.head()

/tmp/ipykernel_1396/944555103.py:6: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


100%|██████████| 18.4M/18.4M [00:00<00:00, 22.7MB/s]
/usr/local/lib/python3.12/dist-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1,246 yds,166 yds,Right Rough,4,423
1,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,2,163 yds,18 ft 5 in.,Green,4,423
2,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,3,20 ft 8 in.,2 ft 1 in.,Green,4,423
3,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,4,2 ft 1 in.,0,In Hole,4,423
4,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,2,1,234 yds,303 yds,Tree Outline,5,532


### Check Data

In [ ]:
# Group by player and tournament, then count UNIQUE rounds and holes
data_check = df.groupby(['tournament_id', 'player']).agg(
    rounds_played=('round', 'nunique'),
    unique_holes_played=('hole', 'nunique')
).reset_index()

# A complete tournament means exactly 4 rounds and 18 unique holes
# flag anyone who doesn't match the 72-hole benchmark
is_complete = (data_check['rounds_played'] == 4) & (data_check['unique_holes_played'] == 18)
missing_data_players = data_check[~is_complete]

# Print the results
if missing_data_players.empty:
    print("All players have complete data (4 rounds, 18 holes each).")
else:
    print(f"Found {len(missing_data_players)} players with missing or incomplete data layers:")
    print(missing_data_players.to_string(index=False))

All players have complete data (4 rounds, 18 holes each).


### Change shotDist and toHole to float datapoints

In [ ]:
import re

def parse_golf_distance_to_yards(val):
    """
    Takes the 'shot_dist' pr 'to_hole' value and converts it to a numerical
    yardage (e.g. a value could be 334 yards converted to 334.0 or 5 ft 11 in.
    converted to 1.972).
    """
    # Convert to string and clean up whitespaces
    val = str(val).strip().lower()

    # Handle clean zeros or empty rows
    if val in ['0', '0.0', 'nan', '']:
        return 0.0

    # Check 1: If it's explicitly in yards (e.g., "334 yds")
    if 'yd' in val:
        match = re.search(r'([\d.]+)', val)
        return float(match.group(1)) if match else 0.0

    # Check 2: If it's in feet/inches (e.g., "5 ft 11 in." or "79 ft 5 in.")
    if 'ft' in val or 'in' in val:
        # Extract feet if present
        ft_match = re.search(r'(\d+)\s*ft', val)
        feet = float(ft_match.group(1)) if ft_match else 0.0

        # Extract inches if present
        in_match = re.search(r'(\d+)\s*in', val)
        inches = float(in_match.group(1)) if in_match else 0.0

        # Convert total feet and inches into decimal yards (3 feet in a yard, 36 inches in a yard)
        total_yards = (feet / 3.0) + (inches / 36.0)
        return round(total_yards, 3) # Rounding to 3 decimal places for precision

    # Check 3: Fallback if it's a raw number string without units
    try:
        return float(val)
    except ValueError:
        return 0.0

# Apply the parsing function to both columns
df["shot_dist_yards"] = df["shot_dist"].apply(parse_golf_distance_to_yards)
df["to_hole_yards"] = df["to_hole"].apply(parse_golf_distance_to_yards)

print(df[["shot_dist", "shot_dist_yards", "to_hole", "to_hole_yards"]].head())

     shot_dist  shot_dist_yards      to_hole  to_hole_yards
0      246 yds          246.000      166 yds        166.000
1      163 yds          163.000  18 ft 5 in.          6.139
2  20 ft 8 in.            6.889   2 ft 1 in.          0.694
3   2 ft 1 in.            0.694            0          0.000
4      234 yds          234.000      303 yds        303.000


### Categorize Locations

In [ ]:
# Sort chronologically so shifts align perfectly within each hole
df = df.sort_values(by=['tournament_id', 'round', 'hole', 'player_id', 'shot_number']).copy()

# Get the *previous* landing location (where the current shot is being hit from)
df['shot_started_from'] = df.groupby(['tournament_id', 'round', 'hole', 'player_id'])['location'].shift(1)

# For the very first shot of a hole, the previous location is blank (NaN),
# which means they are hitting from the Tee Box.
df['shot_started_from'] = df['shot_started_from'].fillna('tee')

# Track the holed status as an independent variable
df["is_holed"] = df["location"].str.lower().str.contains("in hole", na=False)

In [ ]:
def categorize_locations(row):
    """
    Categorize a shot based off location type. Takes in a row/shot and returns
    the location type. The shot location types are tee box, approach, or putt.
    """
    start_loc = str(row["shot_started_from"]).lower().strip()
    end_loc = str(row["location"]).lower().strip()

    # Shot 1 on Par 4s/5s is a Drive (On Par 3s, Shot 1 is technically an Approach)
    try:
        shot_num = int(float(row["shot_number"]))
    except (ValueError, TypeError):
        shot_num = None

    # if row["shot_number"] == 1 or str(row["shot_number"]).strip() == "1" or int(float(row["shot_number"])) == 1:
    if shot_num == 1:
        shot_class = "Tee"
        # if row["par"] == 3:
        #     shot_class = "Tee"
        # else:
        #     shot_class = "Tee"
    elif "green" in start_loc:
        shot_class = "Putt"
    else:
        shot_class = "Approach"

    return shot_class

df["shot_type"] = df.apply(categorize_locations, axis=1)

df.head(10)

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee
1,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,2,163 yds,18 ft 5 in.,Green,4,423,163.000,6.139,Right Rough,False,Approach
2,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,3,20 ft 8 in.,2 ft 1 in.,Green,4,423,6.889,0.694,Green,False,Putt
3,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,4,2 ft 1 in.,0,In Hole,4,423,0.694,0.000,Green,True,Putt
309,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,1,280 yds,132 yds,Right Rough,4,423,280.000,132.000,tee,False,Tee
310,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,2,114 yds,62 ft 10 in.,Right Fairway,4,423,114.000,20.944,Right Rough,False,Approach


In [ ]:
def categorize_positions(row):
    """
    Categorize shots based on where the shot starts from. Takes in row/shot and
    returns the shots start position. The positions are tee, putt, fairway,
    rough, sand, and approach.
    """
    start_loc = str(row["shot_started_from"]).lower().strip()
    end_loc = str(row["location"]).lower().strip()

    # Shot 1 on Par 4s/5s is a Drive (On Par 3s, Shot 1 is technically an Approach)
    if row["shot_number"] == 1:
        shot_pos = "tee"
    elif "green" in start_loc:
        shot_pos = "putt"
    elif "fairway" in start_loc:
        shot_pos = "fairway"
    elif "rough" in start_loc:
        shot_pos = "rough"
    elif "bunker" in start_loc:
        shot_pos = "sand"
    else:
        shot_pos = "approach"

    return shot_pos

df["shot_pos"] = df.apply(categorize_positions, axis=1)

df.head(10)

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type,shot_pos
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee,approach
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach,fairway
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt,putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt,putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee,approach
1,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,2,163 yds,18 ft 5 in.,Green,4,423,163.000,6.139,Right Rough,False,Approach,rough
2,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,3,20 ft 8 in.,2 ft 1 in.,Green,4,423,6.889,0.694,Green,False,Putt,putt
3,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,4,2 ft 1 in.,0,In Hole,4,423,0.694,0.000,Green,True,Putt,putt
309,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,1,280 yds,132 yds,Right Rough,4,423,280.000,132.000,tee,False,Tee,approach
310,THE PLAYERS Championship,R2023011,Stewart Cink,20229,1,1,2,114 yds,62 ft 10 in.,Right Fairway,4,423,114.000,20.944,Right Rough,False,Approach,rough


### Drop Shot Number Nan

In [ ]:
# Force shot_number to be numeric, turning rogue text/strings into NaN safely
df["shot_number"] = pd.to_numeric(df["shot_number"], errors="coerce")

# Drop rows where shot_number became NaN to protect the max calculation
df = df.dropna(subset=["shot_number"])

df.head()

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type,shot_pos
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1.0,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee,approach
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2.0,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach,fairway
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3.0,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt,putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4.0,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt,putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1.0,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee,approach


## Helper Functions

In [ ]:
def get_hole_scores(df):
    """
    Calculates hole scores using .max() on the hole. Returns a column of hole
    scores for every shot called 'total_shots'.
    """
    # Group by player, round, and hole, then find the highest shot number
    shots_per_hole = df.groupby(['tournament_id', 'player', 'round', 'hole'])['shot_number'].max().reset_index()

    # Rename the column to make it clear it represents the hole score
    shots_per_hole.rename(columns={'shot_number': 'total_shots'}, inplace=True)
    return shots_per_hole

## Prepare data

In [ ]:
df.head()

,tournament,tournament_id,player,player_id,round,hole,shot_number,shot_dist,to_hole,location,par,hole_yardage,shot_dist_yards,to_hole_yards,shot_started_from,is_holed,shot_type,shot_pos
40935,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,1.0,235 yds,176 yds,Right Fairway Bunker,4,423,235.000,176.000,tee,False,Tee,approach
40936,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,2.0,170 yds,30 ft 5 in.,Green,4,423,170.000,10.139,Right Fairway Bunker,False,Approach,fairway
40937,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,3.0,34 ft 8 in.,3 ft 11 in.,Green,4,423,11.556,1.306,Green,False,Putt,putt
40938,THE PLAYERS Championship,R2023011,Jerry Kelly,8075,1,1,4.0,3 ft 11 in.,0,In Hole,4,423,1.306,0.000,Green,True,Putt,putt
0,THE PLAYERS Championship,R2023011,Ryan Armour,19803,1,1,1.0,246 yds,166 yds,Right Rough,4,423,246.000,166.000,tee,False,Tee,approach


In [ ]:
GROUP_BY_HOLES = ['tournament_id', 'round', 'hole', 'player']

def get_gir_by_shot(df):
    """
    Calculates if a player gets the  green in regulation (gir) grouped by
    shot. Returns a dataframe with the new column 'GIR'.
    """
    green_shots = df[df['location'].str.lower().str.contains('green', na=False)].copy()

    # Find the first shot that hit the green for every player on every hole
    # (Using groupby + min ensures we catch the exact shot number they reached the surface)
    group_by_first_green_shot = GROUP_BY_HOLES.copy()
    group_by_first_green_shot.append('par')
    first_green_shot = green_shots.groupby(group_by_first_green_shot)['shot_number'].min().reset_index()
    first_green_shot.rename(columns={'shot_number': 'shot_reached_green'}, inplace=True)

    # Apply the official GIR condition: shot_reached_green <= (par - 2)
    first_green_shot['GIR'] = first_green_shot['shot_reached_green'] <= (first_green_shot['par'] - 2)

    # return first_green_shot[['player', 'par', 'shot_reached_green', 'GIR']]
    return first_green_shot

def get_gir_percentage_by_shot(df):
    """
    Calculates the percentage a player gets the  green in regulation (gir)
    grouped by hole. Returns a dataframe with the new
    column 'GIR_%'.
    """
    gir = get_gir_by_shot(df)

    # Calculate final GIR %
    player_gir_df = gir.groupby(GROUP_BY_HOLES)['GIR'].mean().reset_index()

    # Get the percentage
    player_gir_df['GIR_%'] = (player_gir_df['GIR'] * 100).round(2)
    player_gir_df = player_gir_df.sort_values(by='GIR_%', ascending=False).reset_index(drop=True)

    # Clean up temporary aggregation column
    player_gir_df.drop(columns=['GIR'], inplace=True)
    return player_gir_df

def get_player_putts_per_gir_by_shot(df):
    """
    The average amount of putts a player takes when the make the green in
    regulation (GIR). Returns a dataframe with a new column 'putts_per_gir'.
    """
    # Get putts
    putts_df = df[df['shot_type'] == 'Putt'].copy()

    # Group by player and round to count their total putts
    round_putts = putts_df.groupby(GROUP_BY_HOLES).size().reset_index(name="total_putts")

    # Get average putts per round
    player_putts_per_round = round_putts.groupby(GROUP_BY_HOLES)["total_putts"].mean().reset_index(name="putts_per_round")
    player_putts_per_round["putts_per_round"] = player_putts_per_round["putts_per_round"].round(2)

    # Get Putts per GIR

    # Count the number of putts taken on every hole
    hole_putts = putts_df.groupby(GROUP_BY_HOLES).size().reset_index(name='hole_putt_count')

    # Get 'GIR' boolean column (True/False)
    gir_df = get_gir_by_shot(df)
    group_by_gir = GROUP_BY_HOLES.copy()
    group_by_gir.append('GIR')
    gir_putts_base = pd.merge(
        gir_df[group_by_gir],
        hole_putts,
        on=GROUP_BY_HOLES,
        how='left'
    )

    # Fill NaN with 0 for holes where they holed out from off the green and took 0 putts
    gir_putts_base['hole_putt_count'] = gir_putts_base['hole_putt_count'].fillna(0)

    # Isolate only the holes where the player successfully made a GIR
    gir_only_holes = gir_putts_base[gir_putts_base['GIR'] == True]

    # Calculate the final average putts per GIR
    # player_putts_per_gir = gir_only_holes.groupby(['tournament_id', 'player']).size().reset_index(name='putts_per_gir') # total counts
    return gir_only_holes.groupby(GROUP_BY_HOLES)['hole_putt_count'].mean().reset_index(name='putts_per_gir') # average

def get_player_stats_shot_scores_df(df):
    """
    Creates a dataframe of player stats using the helper functions defined above
    Returns a merged dataframe of every player stats calculated in the
    helper functions.
    """
    # New players stats df
    player_df = (
        df[["tournament_id", "player", "player_id", "round", "hole", "shot_number", "location", "shot_dist_yards", "hole_yardage", "to_hole_yards"]]
        .drop_duplicates()
        .sort_values(by=["tournament_id", "player", "round", "hole", "shot_number"])
        .reset_index(drop=True)
    )

    # Get drives, approach, short game, putting, and in shot dist
    gir = get_gir_percentage_by_shot(df)
    putts_per_gir = get_player_putts_per_gir_by_shot(df)

    # Get total strokes
    hole_scores = get_hole_scores(df)

    # Merge dataframes
    player_stats = (
        player_df
        .merge(gir, on=["player", "tournament_id", "round", "hole"], how="left")
        .merge(putts_per_gir, on=["player", "tournament_id", "round", "hole"], how="left")
        .merge(hole_scores, on=["player", "tournament_id", "round", "hole"], how="left")
    )
    return player_stats


# get the players championship data
# Using the players championship because have multiple years of data

# Get players championship 2021-2025
train_tour_ids = [f"R{year}011" for year in range(2021, 2026)]
train_years_only_df = df[df["tournament_id"].isin(train_tour_ids)]
test_years_only_df = df[df["tournament_id"] == "R2026011"]

player_stats_shots = get_player_stats_shot_scores_df(train_years_only_df)
player_stats_shots_test = get_player_stats_shot_scores_df(test_years_only_df)
player_stats_shots.head()

,tournament_id,player,player_id,round,hole,shot_number,location,shot_dist_yards,hole_yardage,to_hole_yards,GIR_%,putts_per_gir,total_shots
0,R2023011,Aaron Baddeley,22371,1,1,1.0,Left Intermediate,288.000,423,129.000,100.0,2.0,4.0
1,R2023011,Aaron Baddeley,22371,1,1,2.0,Green,131.000,423,10.194,100.0,2.0,4.0
2,R2023011,Aaron Baddeley,22371,1,1,3.0,Green,11.889,423,1.583,100.0,2.0,4.0
3,R2023011,Aaron Baddeley,22371,1,1,4.0,In Hole,1.583,423,0.000,100.0,2.0,4.0
4,R2023011,Aaron Baddeley,22371,1,2,1.0,Left Rough,255.000,532,282.000,100.0,1.0,4.0


In [ ]:
# dataset for NN
from sklearn.preprocessing import LabelEncoder, StandardScaler

# feature_cols = ['GIR_%', 'putts_per_gir']
feature_cols = ['GIR_%', 'putts_per_gir', 'shot_number', 'hole_yardage', 'to_hole_yards']

# Just Cameron Young's data
cam_young_df = player_stats_shots[player_stats_shots["player"] == "Cameron Young"]
cam_young_df_test = player_stats_shots_test[player_stats_shots_test["player"] == "Cameron Young"]

location_encoder = LabelEncoder()

# Encode locations
all_locations = pd.concat([cam_young_df['location'], cam_young_df_test['location']]).dropna()
location_encoder.fit(all_locations)
cam_young_df['location_encoded'] = location_encoder.transform(cam_young_df['location'])
cam_young_df_test['location_encoded'] = location_encoder.transform(cam_young_df_test['location'])

# Drop rows with missing values in target or features
cam_young_df = cam_young_df.dropna(subset=feature_cols + ['total_shots'])
cam_young_df_test = cam_young_df_test.dropna(subset=feature_cols + ['total_shots'])

# Extract Features
scaler = StandardScaler()
X_train = cam_young_df[feature_cols].values.astype(np.float32)
X_test = cam_young_df_test[feature_cols].values.astype(np.float32)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # Note: transform, not fit_transform on test set!

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)

# Extract Target 1: Continuous Regression Target (distance)
y_train_shots = cam_young_df['shot_dist_yards'].values.astype(np.float32)
y_test_shots = cam_young_df_test['shot_dist_yards'].values.astype(np.float32)

# Scale shots y dataset
y_scaler_shots = StandardScaler()
y_train_scaled_shots = y_scaler_shots.fit_transform(y_train_shots.reshape(-1, 1))
y_train_shots_t = torch.tensor(y_train_scaled_shots, dtype=torch.float32)  # Shape: [N, 1]
y_test_shots_t = torch.tensor(y_test_shots, dtype=torch.float32).unsqueeze(1)    # Shape: [N, 1]

# Extract Target 2: Categorical Classification Target (location)
y_train_loc = cam_young_df['location_encoded'].values.astype(np.int64)
y_test_loc = cam_young_df_test['location_encoded'].values.astype(np.int64)

y_train_loc_t = torch.tensor(y_train_loc, dtype=torch.long)  # Shape: [N]
y_test_loc_t = torch.tensor(y_test_loc, dtype=torch.long)    # Shape: [N]

/tmp/ipykernel_1396/971367723.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cam_young_df['location_encoded'] = location_encoder.transform(cam_young_df['location'])
/tmp/ipykernel_1396/971367723.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cam_young_df_test['location_encoded'] = location_encoder.transform(cam_young_df_test['location'])


## Model

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, verbose=False, delta=0.0):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
            verbose (bool): If True, prints a message for each validation loss improvement.
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
        """
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        self.best_model_weights = None

    def __call__(self, val_loss, model):
        # First epoch setup
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
        # Validation loss did NOT improve enough
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        # Validation loss improved
        else:
            self.best_loss = val_loss
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model weights when validation loss decreases.'''
        if self.verbose:
            print(f"Validation loss decreased. Saving best model weights...")
        self.best_model_weights = model.state_dict().copy()

In [ ]:
class MultiTaskGolfShots(nn.Module):
    """
    Multi task model. 2 heads:
    1. Regression model for predicting shot distance.
    2. Classification model for predicting shot location.
    """
    def __init__(self, input_dim, num_location_classes):
        super(MultiTaskGolfShots, self).__init__()

        # Shared feature extractor layers
        self.shared_dense1 = nn.Linear(input_dim, 64)
        self.shared_dense2 = nn.Linear(64, 32)

        # Head 1: Regression (Shot Distance) -> 1 float output
        self.distance_head = nn.Linear(32, 1)

        # Head 2: Classification (Shot Location) -> N class logits
        self.location_head = nn.Linear(32, num_location_classes)

    def forward(self, x):
        # Shared representation
        x = F.relu(self.shared_dense1(x))
        x = F.relu(self.shared_dense2(x))

        # Output predictions
        distance_pred = self.distance_head(x) # (batch_size, 1)
        location_pred = self.location_head(x) # (batch_size, num_classes)

        return distance_pred, location_pred

In [ ]:
num_location_classes = len(location_encoder.classes_)

epochs = 500
batch_size = 16
model = MultiTaskGolfShots(input_dim=X_train.shape[1], num_location_classes=num_location_classes)

# Loss functions
criterion_distance = nn.MSELoss()
criterion_location = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

In [ ]:
print("--- STARTING TRAINING LOOP ---")
model.train()

early_stopper = EarlyStopping(patience=30, verbose=False) # wait 15 epochs

for epoch in range(epochs):
    num_samples = X_train_t.size(0)

    # Shuffle the dataset indices every epoch
    permutation = torch.randperm(X_train_t.size()[0])
    epoch_loss = 0

    for i in range(0, num_samples, batch_size):
        indices = permutation[i:i+batch_size]
        batch_x = X_train_t[indices]
        batch_y_dist = y_train_shots_t[indices]
        batch_y_loc = y_train_loc_t[indices]

        # 1. Clear out old gradients
        optimizer.zero_grad()

        # 2. Forward Pass
        pred_dist, pred_loc = model(batch_x)
        loss_dist = criterion_distance(pred_dist.squeeze(), batch_y_dist)
        loss_loc = criterion_location(pred_loc, batch_y_loc)

        # # Print raw losses per epoch to inspect balance:
        # if (epoch + 1) % 20 == 0 or epoch == 0:
        #     print(f"Dist Loss: {loss_dist.item():.4f} | Loc Loss: {loss_loc.item():.4f}")

        # Adjust loss weighting if one dwarfs the other:
        loss = (0.1 * loss_dist) + (1.0 * loss_loc)
        # loss = loss_dist + (2.0 * loss_loc)

        # 3. Backward Pass (Calculate gradients)
        loss.backward()

        # 4. Optimize (Update weights)
        optimizer.step()

        epoch_loss += loss.item()

    # Average loss across batches in this epoch
    num_batches = (num_samples + batch_size - 1) // batch_size
    avg_loss = epoch_loss / num_batches

    # Update learning rate scheduler
    scheduler.step(avg_loss)
    current_lr = optimizer.param_groups[0]['lr']

    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] -> Average Loss: {avg_loss:.4f} | LR: {current_lr} | Dist Loss: {loss_dist.item():.4f} | Loc Loss: {loss_loc.item():.4f}")

    # Check early stopping condition
    early_stopper(avg_loss, model)
    if early_stopper.early_stop:
        print(f"\n[!] Early stopping triggered at Epoch {epoch+1}.")
        break

model.load_state_dict(early_stopper.best_model_weights)
print("Loaded best model weights successfully!")

--- STARTING TRAINING LOOP ---
Epoch [1/500] -> Average Loss: 3.5889 | LR: 0.0001 | Dist Loss: 1.2451 | Loc Loss: 3.4592


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([16, 1])) that is different to the input size (torch.Size([16])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([4, 1])) that is different to the input size (torch.Size([4])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Epoch [20/500] -> Average Loss: 2.1421 | LR: 0.0001 | Dist Loss: 0.5318 | Loc Loss: 1.7812
Epoch [40/500] -> Average Loss: 1.6208 | LR: 0.0001 | Dist Loss: 1.2269 | Loc Loss: 2.0642
Epoch [60/500] -> Average Loss: 1.3967 | LR: 0.0001 | Dist Loss: 0.7085 | Loc Loss: 0.2737
Epoch [80/500] -> Average Loss: 1.3070 | LR: 0.0001 | Dist Loss: 0.4488 | Loc Loss: 0.4970
Epoch [100/500] -> Average Loss: 1.2868 | LR: 0.0001 | Dist Loss: 1.1249 | Loc Loss: 1.9790
Epoch [120/500] -> Average Loss: 1.2134 | LR: 0.0001 | Dist Loss: 1.0428 | Loc Loss: 0.4151
Epoch [140/500] -> Average Loss: 1.1912 | LR: 0.0001 | Dist Loss: 0.6939 | Loc Loss: 0.6098
Epoch [160/500] -> Average Loss: 1.1712 | LR: 5e-05 | Dist Loss: 0.4323 | Loc Loss: 0.5916
Epoch [180/500] -> Average Loss: 1.1920 | LR: 5e-05 | Dist Loss: 1.2569 | Loc Loss: 1.8168
Epoch [200/500] -> Average Loss: 1.1770 | LR: 2.5e-05 | Dist Loss: 1.1611 | Loc Loss: 1.4436
Epoch [220/500] -> Average Loss: 1.1607 | LR: 6.25e-06 | Dist Loss: 0.7811 | Loc Loss

## Results

In [ ]:
model.eval()  # Set model to evaluation mode

with torch.no_grad():
    # 1. Unpack the tuple returned by the model
    pred_dist_logits, pred_loc_logits = model(X_test_t)

    # 2. Process Distance Predictions
    pred_dist_logits = pred_dist_logits.cpu().numpy()
    dist_predictions = y_scaler_shots.inverse_transform(pred_dist_logits).squeeze()

    # 3. Process Location Predictions (Logits -> predicted class index)
    loc_predictions = torch.argmax(pred_loc_logits, dim=1)

print("Distance Predictions:", dist_predictions.squeeze())
print("Location Class Predictions:", loc_predictions)

Distance Predictions: [122.43063  106.11889  103.04792  100.85514  115.45825  115.76921
 108.42668   96.34332  109.293    108.84464  100.33679  125.19685
 108.88458  105.90082  123.968544 104.89478  102.11588  100.05803
 122.64647  106.30241  102.33731   99.95356  114.63082  115.09922
 106.9103    97.34546  123.21074  106.472664 102.86605  101.061745
 116.96993  114.49838  101.64603   99.50101   91.89199  122.78592
 109.334984 100.93698  102.49778  110.6896   109.056854 100.35382
 119.1724   105.88281  102.18717   99.15469  110.45545  108.246086
  98.46827  130.64883  113.614944 104.5615   123.42364  107.5444
 103.04537  100.85514  120.19226  105.36541  102.617836  99.44242
 109.26687  108.83875  100.33679  123.36473  108.69102  101.756584
 102.3675   125.62162  106.944725 102.34196   99.95356  114.20658
 102.897156 123.63583  106.78341  102.87137  101.061745 123.63275
 109.50739  101.02777  102.49778  109.29054  108.847176 100.35382
 125.38798  114.23324  105.38076  123.216324 114.017

In [ ]:
# results_shots_regression = player_stats_shots_test.copy()
results_shots_regression = cam_young_df_test.copy()
results_shots_regression['predicted_shot_dist'] = dist_predictions
results_shots_regression['predicted_location'] = location_encoder.inverse_transform(loc_predictions.cpu().numpy())
results_shots_regression.head()

,tournament_id,player,player_id,round,hole,shot_number,location,shot_dist_yards,hole_yardage,to_hole_yards,GIR_%,putts_per_gir,total_shots,location_encoded,predicted_shot_dist,predicted_location
4937,R2026011,Cameron Young,57366,1,1,1.0,Left Fairway,324.000,424,91.000,100.0,2.0,4.0,11,122.430634,Left Fairway
4938,R2026011,Cameron Young,57366,1,1,2.0,Right Green,94.000,424,2.472,100.0,2.0,4.0,24,106.118889,Green
4939,R2026011,Cameron Young,57366,1,1,3.0,Right Green,3.222,424,0.722,100.0,2.0,4.0,24,103.047920,Green
4940,R2026011,Cameron Young,57366,1,1,4.0,In Hole,0.722,424,0.000,100.0,2.0,4.0,6,100.855141,In Hole
4941,R2026011,Cameron Young,57366,1,2,1.0,Left Rough,264.000,555,282.000,100.0,1.0,4.0,16,115.458252,Right Fairway


In [ ]:
# Get shot distance accuracy
avg_abs_error_dist = (results_shots_regression['shot_dist_yards'] - results_shots_regression['predicted_shot_dist']).abs().mean()
print(f"Absolute Error Dist: {avg_abs_error_dist:.2f}")

# Get location accuracy
accuracy_loc = (results_shots_regression['location'] == results_shots_regression['predicted_location']).mean()
error_rate_loc = 1.0 - accuracy_loc
print(f"Location Accuracy: {accuracy_loc * 100:.2f}%")
print(f"Location Error Rate: {error_rate_loc * 100:.2f}%")

Absolute Error Dist: 94.60
Location Accuracy: 36.56%
Location Error Rate: 63.44%


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(
    results_shots_regression['location'],
    results_shots_regression['predicted_location']
))

                    precision    recall  f1-score   support

             Green       0.00      0.00      0.00         0
           In Hole       0.96      0.98      0.97        51
       Left Bunker       0.00      0.00      0.00         5
      Left Fairway       0.43      0.90      0.58        20
        Left Green       0.00      0.00      0.00        48
        Left Rough       0.00      0.00      0.00         3
 Left Tree Outline       0.00      0.00      0.00         2
      Right Bunker       0.00      0.00      0.00         1
     Right Fairway       0.00      0.00      0.00        11
       Right Green       0.00      0.00      0.00        38
Right Intermediate       0.00      0.00      0.00         2
       Right Rough       0.00      0.00      0.00         5

          accuracy                           0.37       186
         macro avg       0.12      0.16      0.13       186
      weighted avg       0.31      0.37      0.33       186



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [ ]:
# Save model
from google.colab import files

filename = 'model_shot_dist_and_locations.pt'
torch.save(model.state_dict(), filename)

files.download(filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>